In [0]:
df_running_cycling = spark.table("garmin_lakehouse.raw.activities") \
    .filter("get_json_object(activityType, '$.typeKey') IN ('running', 'cycling')") \
    .select("activityId", "activityName", "startTimeLocal", "activityType")

print(f"Running + cycling activities: {df_running_cycling.count()}")
df_running_cycling.display()

In [0]:
%pip install garminconnect

In [0]:
import pandas as pd
from garminconnect import Garmin
import json
import time

In [0]:
email = dbutils.secrets.get(scope="garmin", key="email")
password = dbutils.secrets.get(scope="garmin", key="password")
print("Credentials loaded ✅")

In [0]:
api = Garmin(email=email, password=password)
api.login()
print("Garmin API connected ✅")

In [0]:
# Pipeline parameters
# ⚠️ IMPORTANT: Always commit with load_type = "incremental"
load_type = "incremental"

In [0]:
# Get running activity IDs
all_ids = spark.table("garmin_lakehouse.raw.activities") \
    .filter("get_json_object(activityType, '$.typeKey') LIKE '%running%'") \
    .select("activityId")

if load_type == "full":
    activity_ids = [row.activityId for row in all_ids.collect()]
    print(f"Full load: fetching splits for {len(activity_ids)} activities")

else:
    if spark.catalog.tableExists("garmin_lakehouse.raw.splits"):
        existing_ids = spark.table("garmin_lakehouse.raw.splits") \
            .select("activityId") \
            .distinct()
        new_ids = all_ids.subtract(existing_ids)
    else:
        print("Splits table not found — running full load")
        new_ids = all_ids

    activity_ids = [row.activityId for row in new_ids.collect()]
    print(f"Incremental load: fetching splits for {len(activity_ids)} activities")

In [0]:
splits_results = []
skipped = 0
fetched = 0

for activity_id in activity_ids:
    try:
        splits = api.get_activity_splits(activity_id)
        
        if splits and 'lapDTOs' in splits and splits['lapDTOs']:
            for lap in splits['lapDTOs']:
                lap['activityId'] = activity_id
                splits_results.append(lap)
            fetched += 1
        else:
            skipped += 1
    except Exception as e:
        print(f"Error fetching splits for {activity_id}: {e}")
        skipped += 1
    
    time.sleep(0.1)

print(f"Activities with splits: {fetched}")
print(f"Activities without splits: {skipped}")
print(f"Total lap records: {len(splits_results)}")

In [0]:
# Check if there's any data to process
if not splits_results:
    print("✅ No new splits data to process in this incremental run")
    dbutils.notebook.exit("SUCCESS: No new data")

# Convert to DataFrame
df_splits = pd.DataFrame(splits_results)

print(f"Columns: {len(df_splits.columns)}")
print(f"Rows: {len(df_splits)}")

# Serialise complex fields
for col_name in df_splits.columns:
    if df_splits[col_name].apply(lambda x: isinstance(x, (dict, list))).any():
        df_splits[col_name] = df_splits[col_name].apply(
            lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
        )

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(df_splits)

# Write to Delta table
if not spark.catalog.tableExists("garmin_lakehouse.raw.splits"):
    spark_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("garmin_lakehouse.raw.splits")
    print("Table created ✅")
else:
    from delta.tables import DeltaTable
    
    delta_table = DeltaTable.forName(spark, "garmin_lakehouse.raw.splits")
    
    # Build column mappings for only the columns in the new data
    col_mapping = {col: f"new.{col}" for col in spark_df.columns}
    
    delta_table.alias("existing") \
        .merge(
            spark_df.alias("new"),
            "existing.activityId = new.activityId AND existing.lapIndex = new.lapIndex"
        ) \
        .whenMatchedUpdate(set=col_mapping) \
        .whenNotMatchedInsert(values=col_mapping) \
        .execute()
    print("Merge complete ✅")

count = spark.table("garmin_lakehouse.raw.splits").count()
print(f"Total rows: {count}")